# Run the original CS336 project
On your Mac, edit `CS336-hw1/assignment1-basics` in VS Code and sync that folder with Google Drive for desktop. In Drive, add a shortcut to the synced folder in **My Drive** named **assignment1-basics**. Use **Runtime → Change runtime type** to select a GPU.

Run both cells when you connect. After editing a Python or YAML file in VS Code, save the file, wait for Drive to finish syncing, and rerun the final cell.

In [ ]:
%pip -q install "einops>=0.8.1" jaxtyping pyyaml regex wandb tensorboard matplotlib

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/assignment1-basics")
if not (project / "run_experiments.py").is_file():
    raise FileNotFoundError("In My Drive, add an assignment1-basics shortcut to the synced Mac folder containing run_experiments.py.")
print("Running the project from:", project)

In [ ]:
import os
from pathlib import Path
from runpy import run_path
from tempfile import TemporaryDirectory

import torch
import yaml
from IPython.display import Image, display

os.chdir(project)
settings = yaml.safe_load((project / "config/graph_experiments.yaml").read_text())
settings["defaults"]["device"] = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", settings["defaults"]["device"])
print("Experiments:", ", ".join(settings["experiments"]))

# Only the runtime's config copy needs the Colab device setting.
with TemporaryDirectory() as tmp:
    config_path = Path(tmp) / "colab.yaml"
    config_path.write_text(yaml.safe_dump(settings, sort_keys=False))
    compare = run_path(str(project / "run_experiments.py"))["run_comparison"]
    figure = compare(config_path, project / "exp_output/comparisons")
display(Image(filename=str(figure)))